In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import imageio
from xarray.coders import CFDatetimeCoder

# =========================================================
# Utilities
# =========================================================

def compute_edges(coord):
    """Given 1D coord, return cell-edge coords for pcolormesh."""
    d = np.diff(coord) / 2.0
    return np.concatenate([[coord[0] - d[0]], coord[:-1] + d, [coord[-1] + d[-1]]])

def plev_to_hPa(plev_da):
    """
    Convert model plev coord to hPa.
    Assumes:
      - If max(p) > 2000, p is in Pa -> divide by 100.
      - Else assume already hPa.
    Returns xr.DataArray (plev).
    """
    p = plev_da.values
    if np.nanmax(p) > 2000:
        # Pa → hPa
        p_hPa = p / 100.0
    else:
        # already hPa
        p_hPa = p
    return xr.DataArray(p_hPa, coords={'plev': plev_da}, dims=['plev'])

def pressure_to_height(plev, p0=1e5, H=7000.0):
    """
    Rough log-pressure height (km).
    p0 in Pa, H is scale height (m).
    If plev is in hPa (max < 2000), convert to Pa first.
    """
    p = plev.values.astype(float)
    if np.nanmax(p) < 2000:  # probably hPa
        p = p * 100.0
    z = -H * np.log(p / p0) / 1000.0  # km
    return xr.DataArray(z, coords={'plev': plev}, dims=['plev'])

def load_hindcast(pattern, start, end):
    """
    Load hindcast experiment (possibly multiple short files).
    Average over member if multiple files.
    """
    coder = CFDatetimeCoder(use_cftime=True)
    files = sorted(glob.glob(pattern))
    dss = []
    for f in files:
        ds = xr.open_dataset(f, decode_times=coder)
        sel = ds.sel(time=slice(start, end))
        if sel.time.size > 0:
            dss.append(sel)
    if not dss:
        raise RuntimeError(f"No hindcast data for pattern {pattern}")
    ds_all = xr.concat(dss, dim='member')
    ds_all = ds_all.mean(dim='member')
    return ds_all

def load_control(path, start, end):
    """
    Load long control / reference integration,
    slice same time window.
    """
    coder = CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(path, decode_times=coder)
    sel = ds.sel(time=slice(start, end))
    if sel.time.size == 0:
        raise RuntimeError(f"No control data for slice {start}–{end}")
    return sel

# =========================================================
# O3 DU/km conversion
# =========================================================

def compute_o3_dukm_full(ds):
    """
    Convert O3 mixing ratio (vmr, mol/mol) to DU/km using co-located T, plev.
    Formula:
        DU/km = 26.956 * [ P(hPa) / T(K) ] * O3_ppmv
    where O3_ppmv = O3 * 1e6  (if O3 is vmr mol/mol).

    Inputs:
        ds must have: O3(time, plev, lat, lon),
                      T(time, plev, lat, lon),
                      plev(plev)
    Returns:
        o3_dukm(time, plev, lat, lon) in DU/km
    """
    if ('O3' not in ds) or ('T' not in ds) or ('plev' not in ds):
        raise KeyError("Dataset missing O3, T, or plev for DU/km calc.")

    O3 = ds['O3']  # vmr mol/mol?
    T  = ds['T']   # K
    p_hPa = plev_to_hPa(ds['plev'])  # plev -> hPa (plev)

    # broadcast p_hPa to full field
    # xarray broadcasting should work automatically
    O3_ppmv = O3 * 1e6
    COEFF = 26.95560296256018  # from our derivation
    o3_dukm = COEFF * p_hPa * O3_ppmv / T

    o3_dukm.attrs['units'] = 'DU/km'
    o3_dukm.attrs['long_name'] = 'O3 local column density per km'
    return o3_dukm

# =========================================================
# Tropopause height diagnostic
# =========================================================

def _find_lapse_rate_tropopause_single_col(T_prof, z_prof, min_layer_km=2.0):
    """
    One-column helper:
    - T_prof: 1D array [level]
    - z_prof: 1D array [level] in km
    Return approximate lapse-rate tropopause height (km).

    WMO-ish logic:
      1. Sort by height ascending.
      2. Compute lapse rate Γ = -dT/dz (K/km) between levels.
      3. Scan upward: first level i where
             Γ[i] <= 2 K/km  AND
         the mean Γ from i up to i+Δz(~2 km) also <= 2 K/km.
      4. Tropopause height = z[i].
    Fallback:
      If not found, return cold-point height (min T).
    """

    # ensure ascending height
    idx = np.argsort(z_prof)
    z_sorted = z_prof[idx]
    T_sorted = T_prof[idx]

    # finite-only mask
    mask = np.isfinite(z_sorted) & np.isfinite(T_sorted)
    if np.sum(mask) < 3:
        # too few valid levels, just fallback to min T
        i_min = np.nanargmin(T_prof)
        return z_prof[i_min]

    z_sorted = z_sorted[mask]
    T_sorted = T_sorted[mask]

    # vertical spacing
    dz = np.diff(z_sorted)
    dT = np.diff(T_sorted)
    with np.errstate(divide='ignore', invalid='ignore'):
        lapse = -(dT / dz)  # K/km

    # search tropopause criterion
    for i in range(len(lapse)):
        if not np.isfinite(lapse[i]):
            continue
        if lapse[i] <= 2.0:
            z0 = z_sorted[i]
            z_top = z0 + min_layer_km
            # take all layers from i upward until we exceed z_top
            upper_idx = np.where(z_sorted[:-1] >= z0)[0]
            upper_idx = upper_idx[z_sorted[upper_idx] <= z_top]
            if upper_idx.size == 0:
                mean_lapse = lapse[i]
            else:
                # map layer indices in lapserate array
                layer_inds = [k for k in upper_idx if k < len(lapse)]
                if not layer_inds:
                    mean_lapse = lapse[i]
                else:
                    mean_lapse = np.nanmean(lapse[layer_inds])

            if np.isfinite(mean_lapse) and mean_lapse <= 2.0:
                return z0

    # fallback: cold-point
    i_min = np.nanargmin(T_prof)
    return z_prof[i_min]

def compute_tropopause_heights(T2d, height_km, min_layer_km=2.0):
    """
    Estimate tropopause height vs latitude using lapse-rate method.
    If criterion fails, fallback to cold-point.

    Inputs:
        T2d: xr.DataArray (plev, lat) temperature [K]
        height_km: xr.DataArray (plev) height [km]
    Output:
        tropo_height(lat) [km], np.ndarray
    """
    T_vals = T2d.values  # (plev, lat)
    z_vals = height_km.values  # (plev,)
    nlat = T_vals.shape[1]
    tropo = np.full(nlat, np.nan)

    for j in range(nlat):
        T_prof = T_vals[:, j]
        tropo[j] = _find_lapse_rate_tropopause_single_col(
            T_prof, z_vals, min_layer_km=min_layer_km
        )

    return tropo  # shape (lat,)

# =========================================================
# Plotting / GIF routines
# =========================================================

def plot_spliced_seasonal_average(datasets, lat, height_km, output_file):
    """
    Seasonal-mean style panel plot:
    - Color: O3 in DU/km (time+lon mean)
    - Contours: U (20,30,40,50 m/s) time+lon mean
    - Black curve: tropopause height from lapse-rate method
    """
    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)
    tags = list(datasets.keys())

    fig, axes = plt.subplots(2, 3, figsize=(15, 10),
                             sharex=True, sharey=True)
    axes = axes.flatten()

    for i, tag in enumerate(tags):
        ds = datasets[tag]

        # --- O3 DU/km ---
        o3_dukm = compute_o3_dukm_full(ds)  # (time,plev,lat,lon)
        o3_mean = o3_dukm.mean(dim=['time', 'lon'])  # (plev,lat)

        # wind climatology:
        u_mean  = ds['U'].mean(dim=['time', 'lon'])  # (plev,lat)

        # T mean for tropopause:
        Tm = ds['T'].mean(dim=['time', 'lon'])       # (plev,lat)
        tropo = compute_tropopause_heights(Tm, height_km)  # (lat,)

        ax = axes[i]
        pcm = ax.pcolormesh(
            LAT_e, HT_e, o3_mean.values,
            shading='auto', cmap='rainbow'
        )

        ct  = ax.contour(
            LATc, HTc, u_mean.values,
            levels=[20,30,40,50],
            colors='white', linewidths=1
        )
        ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

        # tropopause line
        ax.plot(lat, tropo, color='k', linewidth=2)

        ax.set_title(f'({chr(97+i)}) {tag}')
        ax.set_xlabel('Latitude (°)')
        ax.set_ylabel('Height (km)')
        ax.set_ylim(0, 30)

    # remove any unused last panel if #tags<6
    if len(tags) < len(axes):
        for k in range(len(tags), len(axes)):
            fig.delaxes(axes[k])

    # shared colorbar
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(
        pcm, cax=cax, orientation='vertical',
        label='O₃ [DU/km]'
    )
    plt.tight_layout(rect=[0, 0, 0.9, 1])
    fig.savefig(output_file, dpi=200)
    plt.close(fig)


def create_o3_gif(ds, lat, height_km, frames_dir, gif_file, duration=0.5):
    """
    Single-scenario animated Hovmöller-style cross-sections:
    - Color: O3 in DU/km
    - Contours: U
    - Tropopause: black line
    Uses a fixed color scale across frames for consistency.
    """
    os.makedirs(frames_dir, exist_ok=True)

    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    # Precompute O3 DU/km (time,plev,lat,lon)
    o3_dukm_full = compute_o3_dukm_full(ds)
    # We'll plot lon-mean at each time step
    o3_dukm_lonmean = o3_dukm_full.mean(dim='lon')  # (time,plev,lat)

    # global color scale for this dataset
    vmin = float(np.nanmin(o3_dukm_lonmean.values))
    vmax = float(np.nanmax(o3_dukm_lonmean.values))

    writer = imageio.get_writer(gif_file, mode='I', duration=duration)

    for i, t in enumerate(ds.time.values):
        # slice this timestep
        o3_now = o3_dukm_lonmean.isel(time=i)        # (plev,lat)
        u_now  = ds['U'].isel(time=i).mean(dim='lon')# (plev,lat)
        T_now  = ds['T'].isel(time=i).mean(dim='lon')# (plev,lat)
        tropo  = compute_tropopause_heights(T_now, height_km)

        fig, ax = plt.subplots(figsize=(8, 6))
        pcm = ax.pcolormesh(
            LAT_e, HT_e, o3_now.values,
            shading='auto', cmap='rainbow',
            vmin=vmin, vmax=vmax
        )
        ct = ax.contour(
            LATc, HTc, u_now.values,
            levels=[20,30,40,50],
            colors='white', linewidths=1
        )
        ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

        ax.plot(lat, tropo, color='k', linewidth=2)

        ax.set_title(str(t))
        ax.set_xlabel('Latitude (°)')
        ax.set_ylabel('Height (km)')
        ax.set_ylim(0, 30)

        # Optional: add colorbar each frame (keeps units explicit in GIF)
        cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
        cb = fig.colorbar(pcm, cax=cax, orientation='vertical')
        cb.set_label('O₃ [DU/km]')

        frame = os.path.join(frames_dir, f'frame_{i:03d}.png')
        fig.savefig(frame, dpi=150)
        plt.close(fig)

        writer.append_data(imageio.imread(frame))

    writer.close()


def create_combined_gif(datasets, lat, height_km,
                        frames_dir, combo_file,
                        duration=0.5):
    """
    Multi-panel animated comparison:
    - Each panel is one dataset
    - Color: O3 DU/km (lon mean)
    - Contours: U (lon mean)
    - Tropopause: black line
    - Shared colorbar per frame (consistent vmin/vmax across all datasets and all times)
    """
    os.makedirs(frames_dir, exist_ok=True)

    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    tags = list(datasets.keys())

    # Precompute DU/km and lon-mean for every dataset
    dukm_lonmean = {}
    for tag, ds in datasets.items():
        o3_dukm_full = compute_o3_dukm_full(ds)
        dukm_lonmean[tag] = o3_dukm_full.mean(dim='lon')  # (time,plev,lat)

    # Determine common time axis (assume same length/order)
    times = next(iter(datasets.values())).time.values

    # Global color limits across ALL datasets and ALL frames
    all_vals = []
    for tag in tags:
        all_vals.append(dukm_lonmean[tag].values)
    all_vals = np.concatenate([a.reshape(-1) for a in all_vals])
    vmin = float(np.nanmin(all_vals))
    vmax = float(np.nanmax(all_vals))

    writer = imageio.get_writer(combo_file, mode='I', duration=duration)

    for ti, t in enumerate(times):
        fig, axes = plt.subplots(2, 3, figsize=(15, 10),
                                 sharex=True, sharey=True)
        axes = axes.flatten()

        for j, tag in enumerate(tags):
            ds = datasets[tag]
            o3_now = dukm_lonmean[tag].isel(time=ti)       # (plev,lat)
            u2d    = ds['U'].isel(time=ti).mean(dim='lon') # (plev,lat)
            Tm     = ds['T'].isel(time=ti).mean(dim='lon') # (plev,lat)
            tropo  = compute_tropopause_heights(Tm, height_km)

            ax = axes[j]
            pcm = ax.pcolormesh(
                LAT_e, HT_e, o3_now.values,
                shading='auto', cmap='rainbow',
                vmin=vmin, vmax=vmax
            )
            ct  = ax.contour(
                LATc, HTc, u2d.values,
                levels=[20,30,40,50],
                colors='white', linewidths=1
            )
            ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

            ax.plot(lat, tropo, color='k', linewidth=2)

            ax.set_title(f'({chr(97+j)}) {tag}')
            ax.set_xlabel('Latitude (°)')
            ax.set_ylabel('Height (km)')
            ax.set_ylim(0, 30)

        # drop any unused last axis if <6 tags
        if len(tags) < len(axes):
            for k in range(len(tags), len(axes)):
                fig.delaxes(axes[k])

        # shared colorbar
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        cb = fig.colorbar(pcm, cax=cax, orientation='vertical')
        cb.set_label('O₃ [DU/km]')

        fig.suptitle(str(t))

        plt.tight_layout(rect=[0, 0, 0.9, 0.95])
        frame = os.path.join(frames_dir, f'combo_{ti:03d}.png')
        fig.savefig(frame, dpi=150)
        plt.close(fig)

        writer.append_data(imageio.imread(frame))

    writer.close()


def create_spliced_difference_gif(datasets, lat, height_km,
                                  frames_dir, gif_file,
                                  duration=0.5):
    """
    Animated difference vs 'Reference':
    - Color: ΔO3 DU/km (experiment - Reference), lon-mean
    - Contours: U of experiment
    - Tropopause: from experiment
    - Symmetric color scale (±max_abs)
    """
    os.makedirs(frames_dir, exist_ok=True)

    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    # Which datasets are experiments?
    exp_tags = [t for t in datasets if t != 'Reference']

    # Precompute DU/km lon-mean for each dataset
    dukm_lonmean = {}
    for tag, ds in datasets.items():
        o3_dukm_full = compute_o3_dukm_full(ds)
        dukm_lonmean[tag] = o3_dukm_full.mean(dim='lon')  # (time,plev,lat)

    times = datasets[exp_tags[0]].time.values

    # global symmetric color scale for all diffs/time
    diffs_flat = []
    for tag in exp_tags:
        diff_all = dukm_lonmean[tag] - dukm_lonmean['Reference']
        diffs_flat.append(diff_all.values.reshape(-1))
    diffs_flat = np.concatenate(diffs_flat)
    max_abs = float(np.nanmax(np.abs(diffs_flat)))
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0
    vmin, vmax = -max_abs, max_abs

    writer = imageio.get_writer(gif_file, mode='I', duration=duration)

    for ti, t in enumerate(times):
        fig, axes = plt.subplots(2, 2, figsize=(12, 8),
                                 sharex=True, sharey=True)
        axes = axes.flatten()

        for j, tag in enumerate(exp_tags):
            ds       = datasets[tag]
            diff2d   = (dukm_lonmean[tag].isel(time=ti)
                        - dukm_lonmean['Reference'].isel(time=ti))
            u2d      = ds['U'].isel(time=ti).mean(dim='lon')
            Tm       = ds['T'].isel(time=ti).mean(dim='lon')
            tropo    = compute_tropopause_heights(Tm, height_km)

            ax = axes[j]
            pcm = ax.pcolormesh(
                LAT_e, HT_e, diff2d.values,
                shading='auto', cmap='RdBu_r',
                vmin=vmin, vmax=vmax
            )
            ct  = ax.contour(
                LATc, HTc, u2d.values,
                levels=[20,30,40,50],
                colors='white', linewidths=1
            )
            ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

            ax.plot(lat, tropo, color='k', linewidth=2)

            ax.set_title(f'({chr(97+j)}) {tag} – Reference')
            ax.set_xlabel('Latitude (°)')
            ax.set_ylabel('Height (km)')
            ax.set_ylim(0, 30)

        # shared colorbar
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        cb = fig.colorbar(pcm, cax=cax, orientation='vertical')
        cb.set_label('ΔO₃ [DU/km]')

        fig.suptitle(str(t))
        plt.tight_layout(rect=[0, 0, 0.9, 0.95])

        frame = os.path.join(frames_dir, f'diff_{ti:03d}.png')
        fig.savefig(frame, dpi=150)
        plt.close(fig)

        writer.append_data(imageio.imread(frame))

    writer.close()

# =========================================================
# Main workflow
# =========================================================

if __name__ == "__main__":
    year = '0008'
    start, end = "0008-03-01", "0008-05-31"

    PARENT = "/home/weiji/restart_exam/plots/O3_DUkm"  # changed output dir name
    os.makedirs(PARENT, exist_ok=True)

    patterns = [
        (f"{year}_Feb_couple",
         f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
         f"BWCN.e122.f19_g16.002_{year}/Feb/"
         f"BWCN.e122.f19_g16.002.{year}-02.*.nc"),
        (f"{year}_Mar_couple",
         f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
         f"BWCN.e122.f19_g16.002_{year}/Mar/"
         f"BWCN.e122.f19_g16.002.{year}-03.*.nc"),
        (f"{year}_Feb_nocouple",
         f"/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/"
         f"{year}-02/*.nc"),
        (f"{year}_Mar_nocouple",
         f"/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/"
         f"{year}-03/*.nc"),
    ]

    control_file = (
        "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
        "BWCN.e122.f19_g16.002/"
        "BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
    )

    # Load experiments + reference
    datasets = { tag: load_hindcast(pat, start, end) for tag, pat in patterns }
    datasets['Reference'] = load_control(control_file, start, end)

    # Take common grid info from any dataset
    example = next(iter(datasets.values()))
    lat     = example.lat.values
    plev    = example.plev
    height  = pressure_to_height(plev)  # km

    # 1) Seasonal mean style multi-panel snapshot
    plot_spliced_seasonal_average(
        datasets, lat, height,
        os.path.join(PARENT, "MAM_average_spliced.png")
    )

    # 2) Individual scenario GIFs
    for tag, ds in datasets.items():
        scen_dir = os.path.join(PARENT, tag)
        frames   = os.path.join(scen_dir, "frames")
        gif_file = os.path.join(scen_dir, f"O3_{tag}.gif")
        os.makedirs(scen_dir, exist_ok=True)
        create_o3_gif(ds, lat, height, frames, gif_file)

    # 3) Combined GIF (all datasets in subplots)
    combo_frames = os.path.join(PARENT, "combined_frames")
    combo_gif    = os.path.join(PARENT, "O3_combined.gif")
    create_combined_gif(datasets, lat, height,
                        combo_frames, combo_gif, duration=0.5)

    # 4) Difference GIF (experiment - Reference) in DU/km
    diff_frames = os.path.join(PARENT, "diff_frames")
    diff_gif    = os.path.join(PARENT, "O3_diff_combined.gif")
    create_spliced_difference_gif(datasets, lat, height,
                                  diff_frames, diff_gif, duration=0.5)


In [ ]:
def create_spliced_difference_gif(datasets, lat, height,
                                  frames_dir, gif_file,
                                  duration=0.5):
    """
    为 4 个 hindcast 场景生成 (O3_exp − O3_ref) 差值的 2×2 面板 GIF，
    每帧叠加对应场景的 U 等值线 (20/30/40/50 m/s)，
    差值单位转换为 ppmv（×1e6），色标范围固定为 [-1, 1] ppmv。
    """
    os.makedirs(frames_dir, exist_ok=True)
    # 构造网格边界
    lat_e, h_e = compute_edges(lat), compute_edges(height)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height)

    # 只保留四个 hindcast 场景
    exp_tags = [t for t in datasets if t != 'Reference']
    times    = datasets[exp_tags[0]].time.values

    writer = imageio.get_writer(gif_file, mode='I', duration=duration)
    for i, t in enumerate(times):
        fig, axes = plt.subplots(2, 2, figsize=(12, 8),
                                 sharex=True, sharey=True)
        axes = axes.flatten()

        for j, tag in enumerate(exp_tags):
            ds     = datasets[tag]
            o3_exp = ds['O3'].isel(time=i)
            o3_ref = datasets['Reference']['O3'].isel(time=i)
            # 差值并转换为 ppmv
            diff2d = (o3_exp - o3_ref).mean(dim='lon') * 1e6  
            u2d    = ds['U'].isel(time=i).mean(dim='lon')

            ax = axes[j]
            pcm = ax.pcolormesh(
                LAT_e, HT_e, diff2d.values,
                shading='auto', cmap='RdBu_r',
                vmin=-1, vmax=1
            )
            ct = ax.contour(
                LATc, HTc, u2d.values,
                levels=[20, 30, 40, 50],
                colors='white', linewidths=1
            )
            ax.clabel(ct, fmt='%d', inline=True, fontsize=8)
            ax.set_title(f'({chr(97+j)}) {tag} – Reference')
            ax.set_ylim(0, 30)
            ax.set_xlabel('Latitude (°)')
            ax.set_ylabel('Height (km)')

        plt.tight_layout(rect=[0, 0, 0.9, 1])
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        fig.colorbar(pcm, cax=cax, orientation='vertical',
                     label='ΔO3 (ppmv)')

        frame = os.path.join(frames_dir, f'diff_{i:03d}.png')
        fig.savefig(frame)
        plt.close(fig)
        writer.append_data(imageio.imread(frame))
    writer.close()




# 差值 GIF
diff_frames = os.path.join(PARENT, 'diff_frames')
diff_gif    = os.path.join(PARENT, 'O3_diff_combined.gif')
create_spliced_difference_gif(
    datasets, lat, height,
    diff_frames, diff_gif,
    duration=0.5
)



In [ ]:
# —— Cold Point Tropopause (CPT) 曲线绘制 —— #

# 1. 选取 Feb_couple 第一个时刻数据
tag = f"{year}_Feb_couple"
ds0 = datasets[tag].isel(time=0)

# 2. 计算 lon 平均的 O3 (ppmv)、U 和 T
O3_ppmv = ds0['O3'].mean(dim='lon')  * 1e6     # mol/mol → ppmv
U_lon   = ds0['U'].mean(dim='lon')
T_lon   = ds0['T'].mean(dim='lon')

# 3. 找到每个纬度上的温度最小层（Cold Point Tropopause）
#    T_lon 维度顺序 (plev, lat)
t_vals = T_lon.values                # shape (nplev, nlat)
cpt_idx = np.nanargmin(t_vals, axis=0)   # 每列最小值的索引
# height 是 pressure_to_height 计算出的 DataArray (plev)
h_vals = height.values               # shape (nplev,)
cpt_heights = h_vals[cpt_idx]        # shape (nlat,)

# 4. 构造绘图网格
lat_e, h_e = compute_edges(lat), compute_edges(height)
LAT_e, HT_e = np.meshgrid(lat_e, h_e)
LATc, HTc   = np.meshgrid(lat, height)

# 5. 绘制 Lat–Altitude 背景图：O3_ppmv + U 等值线 + CPT 曲线
fig, ax = plt.subplots(figsize=(10, 6))
# O3 背景填色
pcm = ax.pcolormesh(
    LAT_e, HT_e, O3_ppmv.values,
    shading='auto', cmap='rainbow'
)
# U 等值线
ct = ax.contour(
    LATc, HTc, U_lon.values,
    levels=[20, 30, 40, 50],
    colors='white', linewidths=1
)
ax.clabel(ct, fmt='%d', inline=True, fontsize=8)
# CPT 曲线
ax.plot(lat, cpt_heights,
        color='k', linewidth=2, label='Cold-Point Tropopause')
ax.legend(loc='upper right')

# 6. 坐标、色标和保存
ax.set_xlabel('Latitude (°)')
ax.set_ylabel('Height (km)')
ax.set_ylim(0, 30)
cbar = fig.colorbar(pcm, ax=ax, orientation='vertical', pad=0.02)
cbar.set_label('O₃ mixing ratio (ppmv)')
fig.tight_layout()
fig.savefig(os.path.join(PARENT, f"{tag}_CPT.png"))
plt.show(fig)
